# Agrupamiento de países según indicadores socioeconómicos

Esta libreta aplica agrupamiento no supervisado a un conjunto de indicadores
socioeconómicos de 167 países: mortalidad infantil, exportaciones, gasto en salud,
importaciones, ingreso, inflación, esperanza de vida, tasa de fertilidad y PIB per
cápita. El objetivo es identificar países con perfiles de desarrollo similares, sin
usar ninguna etiqueta de "grupo verdadero".

La libreta tiene tres partes:

- **Parte 1** ajusta K-medias eligiendo un número de grupos sin ningún criterio todavía.
- **Parte 2** retoma el mismo ajuste, ahora con medidas cuantitativas para decidir
  cuántos grupos usar.
- **Parte 3** demuestra, sobre el mismo conjunto de datos, seis métodos adicionales de
  análisis no supervisado: agrupamiento por densidad, agrupamiento jerárquico, mezclas
  de gaussianas, componentes principales, detección de anomalías y mapas
  autoorganizados.

## Preparación

Se cargan las bibliotecas y el conjunto de datos que se utilizarán en el resto de la
libreta.

In [ ]:
# ============================================================
# PREPARACIÓN — no modifiquen esta celda
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

pd.set_option("display.max_columns", None)

Cargamos el conjunto de datos desde el repositorio y revisamos sus primeras filas.

In [ ]:
# ============================================================
# PREPARACIÓN — no modifiquen esta celda
# ============================================================
url = "https://raw.githubusercontent.com/noe-tec/machine-learning/main/datasets/country_data.csv"
df = pd.read_csv(url)

print(df.shape)
df.head()

Revisamos los tipos de dato y los estadísticos descriptivos de cada columna numérica.

In [ ]:
print(df.dtypes)
df.describe()

Adviertan la enorme diferencia de escala entre columnas: `income` llega hasta
125,000 y `gdpp` hasta 105,000, mientras que `total_fer` no supera 7.5 y `health` no
supera 18. Un algoritmo basado en distancias, como K-medias, trataría estas
diferencias de escala como si fueran diferencias reales de importancia si no se
corrigen antes de agrupar.

Además, no todas las columnas están en las mismas unidades: `exports`, `imports` y
`health` están expresadas como porcentaje del PIB per cápita, no como montos
absolutos, y `child_mort` es la mortalidad de menores de cinco años por cada 1,000
nacidos vivos. Tenerlo presente evita interpretar mal los centroides más adelante.

## Parte 1 Ajustar K-medias

*Tiempo estimado: 15 minutos.*

Ajustarán un modelo de K-medias sobre el conjunto de países. En este punto no cuentan
con ningún criterio para elegir el número de grupos: elijan uno, ajusten el modelo y
observen el resultado. La celda final de esta parte les pedirá justificar su
elección.

Separamos el nombre del país antes de cualquier operación numérica: es una
etiqueta, no una variable de agrupamiento, y no debe entrar al escalador ni al
algoritmo. La conservamos aparte para identificar después qué países caen en cada
grupo.

In [ ]:
paises = df["country"].values
X = df.drop(columns=["country"]).values

X.shape

K-medias agrupa con base en distancias euclidianas. Sin escalar, columnas medidas
en miles (`income`, `gdpp`) dominarían por completo sobre columnas medidas en unidades
(`total_fer`). Estandaricen `X` con `StandardScaler` antes de continuar.

In [ ]:
# TODO: creen un StandardScaler, ajústenlo a X con fit_transform y guarden el resultado en X_esc
scaler = None
X_esc = None

Ajusten un modelo `KMeans` sobre `X_esc` con el número de grupos `k` que ustedes
elijan. No existe todavía un criterio correcto para esta elección: usen el que les
parezca razonable y regístrenlo, porque se les pedirá justificarlo.

In [ ]:
# TODO: elijan un valor de k y ajusten KMeans(n_clusters=k, random_state=42, n_init=10) sobre X_esc
k = None
kmeans = None

Para visualizar los grupos en dos dimensiones proyectamos con PCA. Esta
proyección es solo para graficar: el agrupamiento se realizó sobre las nueve
variables escaladas, no sobre las componentes principales.

In [ ]:
def graficar_grupos(X_esc, etiquetas, titulo):
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X_esc)
    n_grupos = len(np.unique(etiquetas))
    cmap = plt.get_cmap("tab10", n_grupos)
    plt.figure(figsize=(7, 5))
    dispersion = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=etiquetas, cmap=cmap, s=40)
    plt.xlabel("Componente principal 1")
    plt.ylabel("Componente principal 2")
    plt.title(titulo)
    plt.colorbar(dispersion, label="Grupo", ticks=range(n_grupos))
    plt.show()

graficar_grupos(X_esc, kmeans.labels_, f"K-medias con k={k} (proyección PCA)")

`kmeans.cluster_centers_` está expresado en unidades estandarizadas (z): un valor
de 0.69 no es un ingreso ni una tasa de fertilidad. Recuperen los centroides en las
unidades originales con `scaler.inverse_transform` y armen un DataFrame con una fila
por grupo.

In [ ]:
# TODO: recuperen los centroides con scaler.inverse_transform(kmeans.cluster_centers_)
# y arme un DataFrame con las columnas originales
centroides = None
centroides_df = None

centroides_df

Listamos los países que integran cada grupo, según la etiqueta asignada por K-medias.

In [ ]:
resultados = pd.DataFrame({"country": paises, "grupo": kmeans.labels_})

for g in sorted(resultados["grupo"].unique()):
    integrantes = resultados.loc[resultados["grupo"] == g, "country"].tolist()
    muestra = ", ".join(integrantes[:15])
    sufijo = f" (+{len(integrantes) - 15} más)" if len(integrantes) > 15 else ""
    print(f"Grupo {g} ({len(integrantes)} países): {muestra}{sufijo}")

**Para responder:** ¿qué valor de `k` eligieron y con qué criterio? ¿Cómo
podrían saber, con lo visto hasta ahora, si su elección es razonable?

## Parte 2 — Elegir el número de grupos

*Tiempo estimado: 15 minutos.*

Se continúa sobre `X_esc` ya calculado en la Parte 1; no es necesario recargar los
datos ni volver a escalar. Esta parte introduce medidas cuantitativas para orientar la
elección de `k`.

El método del codo grafica la inercia (suma de distancias al cuadrado de cada
punto a su centroide) para distintos valores de `k`. Calculen la inercia de un
`KMeans` para cada `k` de 2 a 10.

In [ ]:
rango_k = range(2, 11)
inercias = []

for k_prueba in rango_k:
    modelo = KMeans(n_clusters=k_prueba, random_state=42, n_init=10).fit(X_esc)
    # TODO: agreguen la inercia de "modelo" (atributo .inertia_) a la lista inercias

plt.figure(figsize=(7, 5))
plt.plot(list(rango_k), inercias, marker="o")
plt.xlabel("Número de grupos (k)")
plt.ylabel("Inercia")
plt.title("Método del codo")
plt.show()

El coeficiente de silueta compara, para cada país, qué tan cerca está de los
demás integrantes de su grupo frente a los del grupo vecino más próximo. Va de -1 a 1;
valores más altos indican grupos mejor separados. Calculen la silueta promedio
(`silhouette_score`) para cada `k` de 2 a 10.

In [ ]:
siluetas = []

for k_prueba in rango_k:
    modelo = KMeans(n_clusters=k_prueba, random_state=42, n_init=10).fit(X_esc)
    # TODO: agreguen silhouette_score(X_esc, modelo.labels_) a la lista siluetas

plt.figure(figsize=(7, 5))
plt.plot(list(rango_k), siluetas, marker="o", color="darkorange")
plt.xlabel("Número de grupos (k)")
plt.ylabel("Coeficiente de silueta")
plt.title("Curva de silueta")
plt.show()

Calculen dos índices adicionales para el mismo rango de `k`: Calinski-Harabasz
(más alto es mejor) y Davies-Bouldin (más bajo es mejor).

In [ ]:
calinski = []
davies_bouldin = []

for k_prueba in rango_k:
    modelo = KMeans(n_clusters=k_prueba, random_state=42, n_init=10).fit(X_esc)
    # TODO: agreguen calinski_harabasz_score(X_esc, modelo.labels_) a calinski
    # TODO: agreguen davies_bouldin_score(X_esc, modelo.labels_) a davies_bouldin

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].plot(list(rango_k), calinski, marker="o", color="seagreen")
ejes[0].set_xlabel("Número de grupos (k)")
ejes[0].set_ylabel("Índice de Calinski-Harabasz")
ejes[0].set_title("Calinski-Harabasz (mayor es mejor)")
ejes[1].plot(list(rango_k), davies_bouldin, marker="o", color="firebrick")
ejes[1].set_xlabel("Número de grupos (k)")
ejes[1].set_ylabel("Índice de Davies-Bouldin")
ejes[1].set_title("Davies-Bouldin (menor es mejor)")
plt.tight_layout()
plt.show()

Resumimos los cuatro índices en una sola tabla, con el mejor valor de cada columna resaltado.

In [ ]:
resumen = pd.DataFrame({
    "k": list(rango_k),
    "inercia": inercias,
    "silueta": siluetas,
    "calinski_harabasz": calinski,
    "davies_bouldin": davies_bouldin,
})

def resaltar_mejor(columna):
    if columna.name in ("silueta", "calinski_harabasz"):
        objetivo = columna.max()
    elif columna.name == "davies_bouldin":
        objetivo = columna.min()
    else:
        return [""] * len(columna)
    return ["font-weight: bold; background-color: #dff0d8" if v == objetivo else "" for v in columna]

resumen.style.apply(resaltar_mejor, axis=0).format(
    {"inercia": "{:.1f}", "silueta": "{:.3f}", "calinski_harabasz": "{:.1f}", "davies_bouldin": "{:.3f}"}
)

**Para responder:** comparen los cuatro índices. Es probable que no coincidan en
el mismo valor de `k`. ¿Cuál eligen ahora y con qué evidencia? Si hay desacuerdo entre
índices, resuélvanlo de forma explícita y justifiquen la decisión.

## Parte 3 — Demostraciones

Las siguientes secciones muestran, sobre el mismo conjunto de países, seis métodos
adicionales de análisis no supervisado.

### DBSCAN

A diferencia de K-medias, DBSCAN no recibe un número de grupos como parámetro: forma
un grupo donde la densidad de puntos supera un umbral, y marca como ruido a los puntos
que quedan aislados. Sus parámetros son `eps` (radio de vecindad) y `min_samples`
(mínimo de vecinos para considerar denso un punto).

In [ ]:
# kneighbors incluye al propio país como su vecino más cercano (distancia 0);
# con n_neighbors=3, la última columna es la distancia que cubre a los tres
# puntos más cercanos -incluido el propio país-, coherente con min_samples=3.
vecinos = NearestNeighbors(n_neighbors=3).fit(X_esc)
distancias, _ = vecinos.kneighbors(X_esc)
distancias_ordenadas = np.sort(distancias[:, -1])

plt.figure(figsize=(7, 5))
plt.plot(distancias_ordenadas)
plt.xlabel("Países ordenados por distancia")
plt.ylabel("Distancia a los min_samples=3 vecinos más cercanos")
plt.title("Distancia al k-ésimo vecino (para elegir eps)")
plt.show()

La curva muestra un cambio de pendiente entre 1.2 y 1.4 (alrededor del percentil
65 de las distancias). `NearestNeighbors` cuenta al propio país como su vecino más
cercano, igual que `min_samples` en DBSCAN, así que ambos cálculos son consistentes
entre sí. Elegimos `eps=1.2` con `min_samples=3`, en el borde inferior de ese cambio,
y comparamos el resultado contra K-medias en la misma proyección.

In [ ]:
dbscan = DBSCAN(eps=1.2, min_samples=3)
etiquetas_dbscan = dbscan.fit_predict(X_esc)

n_grupos = len(set(etiquetas_dbscan)) - (1 if -1 in etiquetas_dbscan else 0)
n_ruido = int(np.sum(etiquetas_dbscan == -1))
print(f"DBSCAN encontró {n_grupos} grupos y dejó {n_ruido} países sin asignar (ruido).")

X_2d = PCA(n_components=2, random_state=42).fit_transform(X_esc)
cmap_kmeans = plt.get_cmap("tab10", k)
cmap_dbscan = plt.get_cmap("tab10", n_grupos)
es_ruido = etiquetas_dbscan == -1

fig, ejes = plt.subplots(1, 2, figsize=(12, 5))
ejes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=kmeans.labels_, cmap=cmap_kmeans, s=40)
ejes[0].set_title(f"K-medias (k={k})")
ejes[1].scatter(X_2d[es_ruido, 0], X_2d[es_ruido, 1], c="lightgray", marker="x", s=40, label="Ruido")
ejes[1].scatter(X_2d[~es_ruido, 0], X_2d[~es_ruido, 1], c=etiquetas_dbscan[~es_ruido], cmap=cmap_dbscan, s=40)
ejes[1].set_title("DBSCAN")
ejes[1].legend(loc="best")
for eje in ejes:
    eje.set_xlabel("Componente principal 1")
    eje.set_ylabel("Componente principal 2")
plt.tight_layout()
plt.show()

DBSCAN encontró cuatro grupos densos y dejó 41 países como ruido, frente a la
partición completa de K-medias, que fuerza a **todos** los países a pertenecer a algún
grupo. El ruido combina dos perfiles opuestos: economías pequeñas y muy ricas
(Luxemburgo, Singapur, Suiza, Catar, Noruega) y países con condiciones
socioeconómicas extremas (Haití, Nigeria, Sierra Leona, Congo Rep. Dem.); a ambos los
distingue estar lejos de cualquier región densa, no pertenecer a un grupo en
particular.

### Agrupamiento jerárquico

El agrupamiento jerárquico aglomerativo construye un árbol (dendrograma) fusionando
progresivamente los puntos o grupos más cercanos. Usamos enlace completo (*complete
linkage*): la distancia entre dos grupos es la mayor distancia entre cualquier par de
puntos, uno de cada grupo, de modo que dos grupos solo se fusionan cuando **todos**
sus integrantes están razonablemente cerca entre sí. El número de grupos no se fija al
ajustar el árbol, sino al decidir a qué altura cortarlo.

In [ ]:
Z = linkage(X_esc, method="complete")

plt.figure(figsize=(16, 6))
dendrogram(Z, labels=paises, leaf_rotation=90, leaf_font_size=6)
plt.xlabel("País")
plt.ylabel("Distancia (enlace completo)")
plt.title("Dendrograma jerárquico (enlace completo)")
plt.tight_layout()
plt.show()

Cortamos el mismo árbol a dos alturas distintas para mostrar que el número de
grupos resultante depende de dónde se corta, no de cómo se ajustó el árbol.

In [ ]:
grupos_5 = fcluster(Z, t=8, criterion="distance")
grupos_3 = fcluster(Z, t=11, criterion="distance")

print(f"Corte a distancia 8: {len(set(grupos_5))} grupos, tamaños {sorted(np.bincount(grupos_5)[1:], reverse=True)}")
print(f"Corte a distancia 11: {len(set(grupos_3))} grupos, tamaños {sorted(np.bincount(grupos_3)[1:], reverse=True)}")

El mismo árbol produce 5 grupos al cortarlo a distancia 8 —incluida Nigeria como
grupo de un solo país— y solo 3 al cortarlo a distancia 11, donde Nigeria queda
absorbida en el grupo grande. Cortar más arriba no solo reduce el número de grupos:
también decide si un país atípico se trata como su propio grupo o se fuerza dentro de
uno más grande. El árbol no decide esto por sí solo; solo describe el orden en que los
países se van uniendo.

### Mezcla de gaussianas

A diferencia de K-medias, que asigna cada país a un único grupo, una mezcla de
gaussianas (`GaussianMixture`) modela los datos como una combinación de
distribuciones normales y asigna a cada país una **probabilidad** de pertenecer a
cada componente. El número de componentes puede elegirse con el criterio de
información bayesiano (BIC): valores más bajos indican mejor ajuste, penalizando la
complejidad.

In [ ]:
rango_componentes = range(2, 11)
bic_scores = []

for n_c in rango_componentes:
    gmm = GaussianMixture(n_components=n_c, random_state=42).fit(X_esc)
    bic_scores.append(gmm.bic(X_esc))

plt.figure(figsize=(7, 5))
plt.plot(list(rango_componentes), bic_scores, marker="o", color="purple")
plt.xlabel("Número de componentes")
plt.ylabel("BIC")
plt.title("Criterio de información bayesiano (BIC)")
plt.show()

El BIC se minimiza en 3 componentes. Ajustamos ese modelo y observamos la
probabilidad de pertenencia de algunos países concretos, incluyendo uno con la
pertenencia repartida entre dos componentes.

In [ ]:
gmm = GaussianMixture(n_components=3, random_state=42).fit(X_esc)
probabilidades = gmm.predict_proba(X_esc)

paises_ejemplo = ["Germany", "Haiti", "Brazil", "Dominican Republic"]
indices_ejemplo = [np.where(paises == p)[0][0] for p in paises_ejemplo]

pd.DataFrame(probabilidades[indices_ejemplo], index=paises_ejemplo,
             columns=[f"componente_{i}" for i in range(3)]).round(2)

Alemania y Haití quedan asignados casi por completo a un único componente, en
extremos opuestos del desarrollo socioeconómico. República Dominicana, en cambio,
reparte su probabilidad casi a la mitad entre dos componentes: el modelo le asigna
probabilidades similares a ambos, por lo que su pertenencia resulta ambigua entre esos
dos grupos. Esa ambigüedad —no una posición geométrica intermedia— es lo que K-medias
no puede expresar, porque solo entrega una etiqueta dura por país.

### Componentes principales

PCA no agrupa: reduce la dimensionalidad encontrando combinaciones lineales de las
variables originales que capturan la mayor varianza posible. Se usó ya para graficar
en las secciones anteriores; aquí se examina qué tanta información conserva y qué
significan sus primeras componentes.

In [ ]:
pca_completo = PCA(random_state=42).fit(X_esc)
varianza_acumulada = np.cumsum(pca_completo.explained_variance_ratio_)

plt.figure(figsize=(7, 5))
plt.plot(range(1, len(varianza_acumulada) + 1), varianza_acumulada, marker="o")
plt.xlabel("Número de componentes")
plt.ylabel("Varianza explicada acumulada")
plt.title("Varianza explicada acumulada por PCA")
plt.show()

Las dos primeras componentes concentran más de 60% de la varianza total. Para
interpretarlas, examinamos los *loadings*: el peso de cada variable original en cada
componente.

In [ ]:
loadings = pca_completo.components_[:2].T

plt.figure(figsize=(8, 6))
plt.axhline(0, color="gray", linewidth=0.8)
plt.axvline(0, color="gray", linewidth=0.8)
for i, columna in enumerate(df.drop(columns=["country"]).columns):
    plt.arrow(0, 0, loadings[i, 0], loadings[i, 1], head_width=0.02, color="steelblue")
    plt.text(loadings[i, 0] * 1.1, loadings[i, 1] * 1.1, columna, fontsize=9)
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("Loadings de las dos primeras componentes")
plt.show()

La primera componente (46% de la varianza) opone ingreso, esperanza de vida y
PIB per cápita a mortalidad infantil y fertilidad: es, en esencia, un eje de
desarrollo socioeconómico. La segunda (17%) está dominada por exportaciones e
importaciones, y contrasta con el gasto en salud: separa a las economías más abiertas
al comercio del resto, con poca asociación lineal con el ingreso en estos datos (la
correlación entre esta componente e `income` es de apenas 0.03).

### Isolation Forest

Isolation Forest detecta anomalías aislando observaciones mediante particiones
aleatorias sucesivas: los puntos atípicos requieren, en promedio, menos particiones
para quedar aislados. El parámetro `contamination` fija la proporción esperada de
anomalías en los datos.

In [ ]:
iso = IsolationForest(random_state=42, contamination=0.1, n_estimators=200)
iso.fit(X_esc)
puntajes = iso.decision_function(X_esc)  # valores más bajos = más anómalo

anomalias = pd.DataFrame({"country": paises, "puntaje_anomalia": puntajes}).sort_values("puntaje_anomalia")
anomalias.head(10)

El parámetro `contamination` desplaza los puntajes de `decision_function` por una
constante (ajusta el umbral interno), pero no altera el orden relativo de los países:
el ranking de más a menos anómalo se mantiene igual.

In [ ]:
for cont in [0.05, 0.10, 0.15]:
    modelo = IsolationForest(random_state=42, contamination=cont, n_estimators=200).fit(X_esc)
    n_anomalias = int((modelo.predict(X_esc) == -1).sum())
    print(f"contamination={cont}: {n_anomalias} países marcados como anómalos")

Los diez países más anómalos no comparten un solo perfil: incluyen economías
pequeñas y muy ricas (Luxemburgo, Singapur, Catar, Malta) junto con países de
indicadores sanitarios muy pobres (Haití, Sierra Leona). Nigeria y Guinea Ecuatorial
matizan esa lectura: Nigeria tiene una inflación atípicamente alta (104%), y Guinea
Ecuatorial combina un ingreso de 33,700 y un PIB per cápita de 17,100 -valores altos-
con mortalidad infantil elevada y esperanza de vida baja. Isolation Forest señala
perfiles alejados del país típico del conjunto; no explica por sí solo la causa de esa
distancia (pobreza, conflicto, dependencia de un recurso, etc.), que requeriría
variables adicionales para confirmarse.

### Mapas autoorganizados

Un mapa autoorganizado (SOM) proyecta los datos sobre una retícula de neuronas de baja
dimensión, preservando en lo posible la topología del espacio original: países
similares terminan en neuronas cercanas. A diferencia de PCA, la proyección no es
lineal.

In [ ]:
try:
    !pip install minisom -q
except Exception as error:
    print(f"No se pudo instalar minisom automáticamente: {error}")

from minisom import MiniSom

Entrenamos un SOM sobre una retícula de 6×6 neuronas. La U-matrix resume, para
cada neurona, qué tan lejos están sus pesos de los de sus neuronas vecinas (por
defecto, `distance_map` normaliza la suma de esas distancias, no el promedio),
resaltando las fronteras entre regiones del mapa; junto a ella ubicamos cada país en
su neurona ganadora.

In [ ]:
som = MiniSom(x=6, y=6, input_len=X_esc.shape[1], sigma=1.0, learning_rate=0.5, random_seed=42)
som.random_weights_init(X_esc)
som.train_random(X_esc, 5000)

ganadoras = np.array([som.winner(x) for x in X_esc])

fig, ejes = plt.subplots(1, 2, figsize=(13, 6))
ejes[0].pcolor(som.distance_map().T, cmap="bone_r")
ejes[0].set_title("U-matrix")
ejes[0].set_xlabel("Neurona (eje x)")
ejes[0].set_ylabel("Neurona (eje y)")
ejes[1].pcolor(som.distance_map().T, cmap="bone_r", alpha=0.5)
ejes[1].scatter(ganadoras[:, 0] + 0.5, ganadoras[:, 1] + 0.5, s=15, color="crimson")
ejes[1].set_title("Países en su neurona ganadora")
ejes[1].set_xlabel("Neurona (eje x)")
ejes[1].set_ylabel("Neurona (eje y)")
plt.tight_layout()
plt.show()

Las regiones claras de la U-matrix concentran neuronas con vecinos similares
—zonas del mapa donde tienden a caer países de perfil socioeconómico parecido—,
mientras que las regiones oscuras marcan fronteras entre grupos de países más
distintos entre sí. A diferencia de K-medias o DBSCAN, el SOM además intenta preservar
la vecindad del espacio original al proyectarlo: neuronas cercanas en la retícula
tienden a corresponder a países más parecidos entre sí, no solo dentro del mismo
grupo, aunque esa preservación es aproximada y no una garantía exacta.